# Dataset EDA

Run after `scripts/prepare_dataset.py` and `scripts/prepare_drink_waste.py`.

Answers:
- How many images per split?
- Class distribution — any class starved?
- TACO vs Drink Waste — do sources leak into test?
- Bbox size distribution — are boxes too tiny for YOLO to see?
- Sample gallery per class — does the data actually look right?

In [ ]:
# --- setup ---
from pathlib import Path
from collections import Counter, defaultdict
import random
import cv2
import matplotlib.pyplot as plt

ML_DIR = Path.cwd()
if ML_DIR.name == 'notebooks':
    ML_DIR = ML_DIR.parent
DATA = ML_DIR / 'data'
CLASS_NAMES = ['bottle', 'cup', 'can', 'wrapper', 'paper']
SPLITS = ('train', 'val', 'test')
assert DATA.exists(), f'missing {DATA}. Run the prep scripts first.'
print('ml-training dir:', ML_DIR)

## 1. Image counts per split

In [ ]:
for s in SPLITS:
    n = sum(1 for p in (DATA / 'images' / s).glob('*') if p.suffix.lower() in ('.jpg', '.jpeg', '.png'))
    print(f'{s:>5}: {n:>5} images')

## 2. Class distribution

One row per split. Watch for:
- any class with 0 instances → remap bug or exclusion too aggressive
- imbalance worse than 5:1 → consider class weighting or oversampling at training time

In [ ]:
dist = {s: Counter() for s in SPLITS}
for s in SPLITS:
    for lbl in (DATA / 'labels' / s).glob('*.txt'):
        for line in lbl.read_text().strip().splitlines():
            if line:
                dist[s][int(line.split()[0])] += 1

fig, ax = plt.subplots(figsize=(8, 4))
width = 0.25
x = list(range(len(CLASS_NAMES)))
for i, s in enumerate(SPLITS):
    ax.bar([v + (i - 1) * width for v in x], [dist[s].get(c, 0) for c in range(len(CLASS_NAMES))], width, label=s)
ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES)
ax.set_ylabel('instances')
ax.set_title('class distribution per split')
ax.legend()
plt.tight_layout()
plt.show()

print(f"{'class':<10} {'train':>7} {'val':>7} {'test':>7} {'total':>7}")
for c, n in enumerate(CLASS_NAMES):
    t = dist['train'].get(c, 0)
    v = dist['val'].get(c, 0)
    te = dist['test'].get(c, 0)
    print(f'{n:<10} {t:>7} {v:>7} {te:>7} {t+v+te:>7}')

## 3. Source distribution (TACO vs Drink Waste)

Drink Waste images are prefixed with `dw_`; everything else is TACO.
Watch for test split dominated by one source — would make test mAP unrepresentative of real-world performance.

In [ ]:
for s in SPLITS:
    taco, dw = 0, 0
    for p in (DATA / 'images' / s).glob('*'):
        if p.suffix.lower() not in ('.jpg', '.jpeg', '.png'):
            continue
        if p.stem.startswith('dw_'):
            dw += 1
        else:
            taco += 1
    total = taco + dw
    pct_dw = (dw / total * 100) if total else 0
    print(f'{s:>5}: TACO={taco:>5}  DrinkWaste={dw:>5}  (DW is {pct_dw:.0f}%)')

## 4. Bbox size distribution

YOLO struggles with tiny boxes. If the majority of boxes are <2% of image area, you'll get poor recall at the default 640×640 imgsz — consider going to 960 or 1280.

In [ ]:
areas_pct = []  # bbox area as fraction of image area
per_class_areas = defaultdict(list)
for lbl in (DATA / 'labels' / 'train').glob('*.txt'):
    for line in lbl.read_text().strip().splitlines():
        if not line:
            continue
        cls, cx, cy, w, h = map(float, line.split())
        area = w * h  # normalized, so 0..1 = fraction of image
        areas_pct.append(area * 100)
        per_class_areas[int(cls)].append(area * 100)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.hist(areas_pct, bins=40)
ax1.set_xlabel('bbox area (% of image)')
ax1.set_ylabel('count')
ax1.set_title('bbox size distribution (train)')
ax1.axvline(2, color='r', linestyle='--', label='2% (tiny threshold)')
ax1.legend()

box_data = [per_class_areas[c] for c in range(len(CLASS_NAMES)) if per_class_areas[c]]
box_labels = [CLASS_NAMES[c] for c in range(len(CLASS_NAMES)) if per_class_areas[c]]
ax2.boxplot(box_data, labels=box_labels)
ax2.set_ylabel('bbox area (% of image)')
ax2.set_title('bbox size by class')
ax2.set_yscale('log')
plt.tight_layout()
plt.show()

import numpy as np
areas_np = np.array(areas_pct)
print(f'boxes: {len(areas_np)}')
print(f'  median area: {np.median(areas_np):.2f}%')
print(f'  5th pct:     {np.percentile(areas_np, 5):.2f}%')
print(f'  tiny (<2%):  {(areas_np < 2).sum()} ({(areas_np < 2).mean() * 100:.0f}%)')

## 5. Boxes per image

Most YOLO datasets have 1-3 boxes per image. High counts may be scene images; zero-count images (background) help reduce false positives if intentional.

In [ ]:
boxes_per_img = []
for lbl in (DATA / 'labels' / 'train').glob('*.txt'):
    lines = [l for l in lbl.read_text().strip().splitlines() if l]
    boxes_per_img.append(len(lines))

plt.figure(figsize=(7, 3))
plt.hist(boxes_per_img, bins=range(0, max(boxes_per_img) + 2))
plt.xlabel('boxes per image')
plt.ylabel('count')
plt.title('boxes per image (train)')
plt.show()
print(f'empty labels:  {sum(1 for n in boxes_per_img if n == 0)}')
print(f'median:        {int(np.median(boxes_per_img))}')
print(f'max:           {max(boxes_per_img)}')

## 6. Image dimensions

Ultralytics resizes to `imgsz` (default 640). If your images are mostly 4000×3000 phone shots, you're downsampling 6x — tiny boxes become single-digit pixels. If they're already 640×480, no problem.

In [ ]:
_candidates = [p for p in (DATA / 'images' / 'train').glob('*') if p.suffix.lower() in ('.jpg', '.jpeg', '.png')]
sample = random.sample(_candidates, min(300, len(_candidates)))
dims = []
for p in sample:
    img = cv2.imread(str(p))
    if img is not None:
        dims.append(img.shape[:2])  # (h, w)
heights = [d[0] for d in dims]
widths = [d[1] for d in dims]
plt.figure(figsize=(6, 6))
plt.scatter(widths, heights, alpha=0.4, s=10)
plt.xlabel('width (px)')
plt.ylabel('height (px)')
plt.title(f'image dimensions (sampled {len(dims)})')
plt.axvline(640, color='r', linestyle='--', alpha=0.5, label='imgsz=640')
plt.axhline(640, color='r', linestyle='--', alpha=0.5)
plt.legend()
plt.show()
print(f'median size: {int(np.median(widths))} × {int(np.median(heights))}')

## 7. Sample gallery per class

4 random training images per class with their boxes drawn. Sanity check that labels match reality.

In [ ]:
imgs_by_class = defaultdict(list)
for lbl in (DATA / 'labels' / 'train').glob('*.txt'):
    classes_in_img = set()
    for line in lbl.read_text().strip().splitlines():
        if line:
            classes_in_img.add(int(line.split()[0]))
    for c in classes_in_img:
        img_candidates = list((DATA / 'images' / 'train').glob(lbl.stem + '.*'))
        if img_candidates:
            imgs_by_class[c].append((img_candidates[0], lbl))

random.seed(1)
cols = 4
fig, axes = plt.subplots(len(CLASS_NAMES), cols, figsize=(cols * 3, len(CLASS_NAMES) * 3))
for row, (c, name) in enumerate(zip(range(len(CLASS_NAMES)), CLASS_NAMES)):
    samples = random.sample(imgs_by_class[c], min(cols, len(imgs_by_class[c]))) if imgs_by_class[c] else []
    for col in range(cols):
        ax = axes[row][col] if len(CLASS_NAMES) > 1 else axes[col]
        ax.axis('off')
        if col >= len(samples):
            continue
        img_path, lbl_path = samples[col]
        img = cv2.cvtColor(cv2.imread(str(img_path)), cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        for line in lbl_path.read_text().strip().splitlines():
            if not line:
                continue
            cls, cx, cy, bw, bh = map(float, line.split())
            x1, y1 = int((cx - bw / 2) * w), int((cy - bh / 2) * h)
            x2, y2 = int((cx + bw / 2) * w), int((cy + bh / 2) * h)
            color = (0, 255, 0) if int(cls) == c else (255, 100, 0)
            cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
            cv2.putText(img, CLASS_NAMES[int(cls)], (x1, max(0, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
        ax.imshow(img)
        if col == 0:
            ax.set_title(name, loc='left', fontsize=12)
plt.tight_layout()
plt.show()

## 8. Health checks

Automated sanity: empty labels, out-of-bounds coordinates, degenerate boxes, unknown class IDs.

In [ ]:
issues = Counter()
examples = defaultdict(list)
for s in SPLITS:
    for lbl in (DATA / 'labels' / s).glob('*.txt'):
        text = lbl.read_text().strip()
        if not text:
            issues['empty_label'] += 1
            examples['empty_label'].append(lbl.name)
            continue
        for line in text.splitlines():
            parts = line.split()
            if len(parts) != 5:
                issues['malformed_line'] += 1
                continue
            cls, cx, cy, w, h = map(float, parts)
            if not (0 <= cls < len(CLASS_NAMES)):
                issues['bad_class_id'] += 1
                examples['bad_class_id'].append((lbl.name, int(cls)))
            if not (0 <= cx <= 1 and 0 <= cy <= 1 and 0 < w <= 1 and 0 < h <= 1):
                issues['coord_out_of_range'] += 1
                examples['coord_out_of_range'].append(lbl.name)
            if w < 0.005 or h < 0.005:
                issues['degenerate_tiny_box'] += 1

if not issues:
    print('no issues')
else:
    for k, v in issues.items():
        print(f'{k}: {v}')
        for ex in examples.get(k, [])[:3]:
            print(f'  e.g. {ex}')